# Training and export template (Google Colab)

**This is a template. It contains no dataset, no results and no tuned values.**
You fill in the TODO cells. Work in this order, and never use the test set to make decisions:

1. Load data (keep confidential data in Colab/Drive, never in GitHub)
2. Split data (for machine data over time, split by TIME, not randomly, to avoid leakage)
3. Train a scikit-learn Pipeline on a **pandas DataFrame**
4. Choose the decision threshold on the **validation** set
5. Evaluate ONCE on the test set and record honestly in docs/model-card.md
6. Export the three artifact files and download them

In [ ]:
# 0. Check versions. The backend's scikit-learn minor version must match this.
import sklearn, pandas, numpy
print('scikit-learn', sklearn.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)

## 1. Load data (TODO)
Mount Google Drive or upload your file. Do NOT commit the dataset to GitHub.

In [ ]:
# TODO: example only
# from google.colab import drive
# drive.mount('/content/drive')
# import pandas as pd
# df = pd.read_csv('/content/drive/MyDrive/<your-folder>/<your-file>.csv')

## 2. Split (TODO)
Decide the split strategy and justify it in your dissertation. Keep `X_*` as DataFrames with named columns.

In [ ]:
# TODO: X_train, X_val, X_test, y_train, y_val, y_test

## 3. Train a Pipeline (TODO)
Put scaling/encoding inside the Pipeline so the backend does not need to repeat it.

In [ ]:
# TODO example shape (replace with your own, justified choice):
# from sklearn.pipeline import Pipeline
# pipeline = Pipeline([('scale', ...), ('model', ...)])
# pipeline.fit(X_train, y_train)

## 4. Choose the decision threshold on VALIDATION data (TODO)
Write down how you chose it (e.g. which metric and why).

In [ ]:
# TODO: chosen_threshold = ...

## 5. Export the artifacts
Run the helper cell first, then the export cell.

In [ ]:
"""Export a trained model for the backend. Designed to run inside Google Colab.

Usage in a Colab cell (after you trained `pipeline` on a pandas DataFrame `X_train`):

    export_artifacts(
        pipeline=pipeline,
        feature_names=list(X_train.columns),
        out_dir="/content/artifacts",
        model_name="my-model-name",
        model_version="v1",
        decision_threshold=0.5,   # replace with the value YOU chose on VALIDATION data
    )

It writes three files: model.joblib, feature_schema.json, model_metadata.json.
Nothing here invents values: ranges and thresholds are only written if you supply them.
"""

import json
from datetime import datetime, timezone
from pathlib import Path


def export_artifacts(
    pipeline,
    feature_names,
    out_dir,
    model_name,
    model_version,
    positive_class=1,
    decision_threshold=None,
    feature_info=None,
    notes=None,
):
    """feature_info (optional): {"feature": {"unit": "...", "min": 0, "max": 100, "description": "..."}}
    Only add min/max if they come from your data or a domain expert. Leave them out otherwise."""
    import joblib

    feature_names = list(feature_names)
    feature_info = feature_info or {}

    if not hasattr(pipeline, "predict_proba"):
        raise ValueError("The model must support predict_proba(). Use a classifier that provides probabilities.")

    trained_names = getattr(pipeline, "feature_names_in_", None)
    if trained_names is None:
        raise ValueError(
            "The model has no feature_names_in_. Train it on a pandas DataFrame (not a NumPy array) "
            "so column names and order are recorded."
        )
    if list(trained_names) != feature_names:
        raise ValueError(f"feature_names do not match training columns.\nTrained on: {list(trained_names)}\nGiven:      {feature_names}")

    unknown = set(feature_info) - set(feature_names)
    if unknown:
        raise ValueError(f"feature_info has names not in feature_names: {sorted(unknown)}")

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    joblib.dump(pipeline, out / "model.joblib")

    features = []
    for name in feature_names:
        spec = {"name": name}
        spec.update({k: v for k, v in feature_info.get(name, {}).items() if k in {"unit", "min", "max", "description"}})
        features.append(spec)
    (out / "feature_schema.json").write_text(json.dumps({"features": features}, indent=2), encoding="utf-8")

    metadata = {
        "model_name": model_name,
        "model_version": model_version,
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "positive_class": positive_class,
        "decision_threshold": decision_threshold,
        "notes": notes,
    }
    (out / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    try:
        import sklearn

        print("scikit-learn used for training:", sklearn.__version__, "(backend must use the same minor version)")
    except ImportError:
        pass
    print("Exported to:", out.resolve())
    for p in sorted(out.iterdir()):
        print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# TODO: fill in, then run.
# export_artifacts(
#     pipeline=pipeline,
#     feature_names=list(X_train.columns),
#     out_dir='/content/artifacts',
#     model_name='YOUR-MODEL-NAME',
#     model_version='v1',
#     decision_threshold=chosen_threshold,
#     feature_info={},   # optional units / evidence-based min,max
# )

## 6. Download
Zip and download, then unzip into `ml/artifacts/` on your laptop.

In [ ]:
# import shutil; shutil.make_archive('/content/artifacts', 'zip', '/content/artifacts')
# from google.colab import files; files.download('/content/artifacts.zip')